# Bootstrap do ambiente — do zero ao ambiente pronto (idempotente)

Recria em **qualquer workspace** tudo que o pipeline e o job assumem que existe. Pode rodar quantas vezes quiser: cada passo é `IF NOT EXISTS` ou pula arquivo já presente.

| Passo | O que faz | Idempotência |
| --- | --- | --- |
| 1. Infraestrutura UC | catálogo + schemas `bronze/silver/gold/governanca/landing` + volumes `landing.files` e `landing.reference` + tabelas de governança | `CREATE ... IF NOT EXISTS` |
| 2. Dados de origem | parquets TLC via `url_template` dos contratos (motor genérico `motores.baixar_da_fonte`) | arquivo existente com tamanho > 0 é pulado (log `ja_existia`) |
| 3. Referência TLC | `taxi_zone_lookup.csv` oficial para `landing/reference/` | sobrescreve com snapshot datado (converge para estado válido) |

Ordem de recriação do ambiente (fluxo do SDD): **este notebook → `databricks bundle deploy` → update do pipeline SDP → job `case_ifood_update_e_qualidade`** (qualidade + observabilidade). Widgets: `catalogo`, `ano`, `meses`.

In [0]:
dbutils.widgets.text("catalogo", "case_ifood", "catalogo UC")
dbutils.widgets.text("ano", "2023", "ano")
dbutils.widgets.text("meses", "01,02,03,04,05", "meses (csv)")

catalogo = dbutils.widgets.get("catalogo")
ano = dbutils.widgets.get("ano")
meses = [m.strip().zfill(2) for m in dbutils.widgets.get("meses").split(",") if m.strip()]

import os
import sys

from pathlib import Path

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
candidatos = [Path(notebook_path), Path("/Workspace") / notebook_path.lstrip("/")]
REPO = None
for candidato in candidatos:
    for raiz in (candidato, *candidato.parents):
        if (raiz / "00_config" / "motores.py").is_file():
            REPO = str(raiz)
            break
    if REPO:
        break
if REPO is None:
    raise FileNotFoundError(f"Raiz do projeto nao localizada a partir do notebook: {notebook_path}")
for sub in ("00_config", "01_ingestao"):
    sys.path.insert(0, f"{REPO}/{sub}")

import motores
from atualizar_taxi_zone_lookup import atualizar as atualizar_lookup

CONTRATOS = motores.listar_contratos()
print(f"Catalogo: {catalogo} | periodo: {ano}-{meses}")
print(f"Contratos de ingestao: {[c['dataset'] for c in CONTRATOS]}")

In [0]:
# 1) Infraestrutura UC — tudo IF NOT EXISTS: roda quantas vezes quiser
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalogo}")
for schema in ("bronze", "silver", "gold", "governanca"):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{schema}")

# Volume de landing: no UC o caminho é catalog.schema.volume —
# /Volumes/case_ifood/landing/files = catálogo case_ifood, schema landing,
# volumes files (parquets) e reference (lookup TLC)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.landing")
for volume in ("files", "reference"):
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalogo}.landing.{volume}")

# Tabelas de governança do projeto (log_ingestao, dq_resultados, obs_metricas)
motores.criar_tabelas_governanca(catalogo, spark)

print("Infraestrutura OK:")
print(f"  catalogo: {catalogo}")
print("  schemas: bronze, silver, gold, governanca, landing")
print("  volumes: landing.files (parquets) e landing.reference (lookup TLC)")
print(f"  tabelas de governança em {catalogo}.governanca")

In [0]:
# 2) Dados de origem — o motor pula arquivo já presente (tamanho > 0) e
#    registra checksum/status em governanca.log_ingestao
import pandas as pd

resultados = []
for contrato in CONTRATOS:
    res = motores.baixar_da_fonte(
        contrato, spark=spark, ano=ano, meses=meses, catalogo=catalogo
    )
    resultados.extend(res)

# 3) Lookup oficial de Taxi Zones (dimensão de referência da gold)
info_lookup = atualizar_lookup(dbutils)

display(pd.DataFrame(resultados))
print(f"Lookup TLC: {info_lookup['rows']} zonas | sha256 {info_lookup['sha256'][:12]}… | {info_lookup['path']}")

In [0]:
# Verificação final: parquets esperados por contrato no landing
import fnmatch

falhas = []
for contrato in CONTRATOS:
    volume = contrato["fonte"]["volume"]
    glob_ = contrato["fonte"]["glob"]
    casados = sorted(f for f in os.listdir(volume) if fnmatch.fnmatch(f, glob_))
    ok = len(casados) >= len(meses)
    if not ok:
        falhas.append(contrato["dataset"])
    print(f"{contrato['dataset']}: {len(casados)} parquet(s) "
          f"{'OK' if ok else f'AVISO: esperava {len(meses)}'}")

if falhas:
    print(f"\n[aviso] revise o log do passo 2 para: {falhas}")
else:
    total = sum(1 for c in CONTRATOS)
    print(f"\nBootstrap OK: infraestrutura + dados + lookup TLC prontos.")
    print("Próximos passos (fluxo do SDD, seção 6):")
    print("  1) databricks bundle deploy -t dev     # cria pipeline + job")
    print("  2) update do pipeline SDP              # materializa bronze/silver/gold")
    print("  3) job case_ifood_update_e_qualidade   # qualidade + observabilidade")